# Momants topic classification

This notebook determines the main topic and subtopic of visitor messages with normalized multilingual embeddings. It does not perform sentiment, intent, or answer analysis. Only the six permitted Momants fields are loaded.

## 1. Import the topic module

In [ ]:
from pathlib import Path
import importlib
import sys

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import momants_topic
importlib.reload(momants_topic)

## 2. Set the paths and event

In [ ]:
CSV_PAD = PROJECT_DIR / "data" / "tests" / "conversations_100.csv"
TOPICS_SEED_PATH = PROJECT_DIR / "data" / "seeds" / "topics_en_v2.csv"
EVENT_ID = "decibel_2026"
OUTPUT_DIRECTORY = PROJECT_DIR / "results"

print(f"Input: {CSV_PAD}")
print(f"Topic seed: {TOPICS_SEED_PATH}")
print(f"Event: {EVENT_ID}")
print(f"Output directory: {OUTPUT_DIRECTORY}")

## 3. Validate without loading the model

This checks the safe Momants loader and filters active v2 labels for the selected event.

In [ ]:
data = momants_topic.laad_momants_csv(CSV_PAD)
visitors = momants_topic.selecteer_bezoekersberichten(data)
topics = momants_topic.laad_onderwerpen(TOPICS_SEED_PATH, EVENT_ID)

print(f"Message rows: {len(data)}")
print(f"Usable visitor messages: {len(visitors)}")
print(f"Conversations: {visitors['conversation_id'].nunique()}")
print(f"Active topic labels: {len(topics)}")
display(topics[['main_topic', 'subtopic', 'description']])

## 4. Classify topics

The multilingual sentence-transformer embeds all active subtopic descriptions and five internal `None` prototypes once. Each visitor message is compared directly with those cached embeddings. The closest subtopic wins unless a `None` prototype is closer; the main topic is then derived from the seed. `similarity` is cosine similarity, not a probability. Messages classified as `None` do not appear in the output.

In [ ]:
topic_results = momants_topic.process_csv(
    csv_path=CSV_PAD,
    seed_path=TOPICS_SEED_PATH,
    event_id=EVENT_ID,
    output_directory=OUTPUT_DIRECTORY,
    batch_size=16,
)

print(f"Complete: {len(topic_results)} conversation-topic combinations found.")
print(f"Conversation output: {topic_results.attrs['output_path']}")
print(f"Validation debug output: {topic_results.attrs['debug_output_path']}")
topic_results.head(20)

## 5. Validation

Validate every visitor-message instance from the privacy-safe debug export against the normalized-text answer key. This reports all six required metrics, the full confusion matrix, prediction counts, and every error.

In [ ]:
import validate_momants_topic

ANSWER_KEY_PATH = PROJECT_DIR / "data" / "tests" / "answer_key_v2.csv"
validation_metrics = validate_momants_topic.valideer(
    csv_pad=None,
    antwoordsleutel_pad=ANSWER_KEY_PATH,
    seed_pad=TOPICS_SEED_PATH,
    event_id=EVENT_ID,
    debug_pad=topic_results.attrs["debug_output_path"],
)
validation_metrics